# Which Wisdom Book Is This Passage From? Supervised Text Classification of Biblical and Asian Religious Texts

**Name:** Arturo Ramos
**Dataset:** *A Study of Asian Religious and Biblical Texts* (UCI Machine Learning Repository, 2019, CC BY 4.0) — https://doi.org/10.24432/C55S4W — file `AllBooks_baseline_DTM_Labelled.csv`

**Problem.** This is a **supervised learning, multi-class classification** task. Each row is a passage (a chapter-sized chunk) from one of eight wisdom books — four biblical (Proverbs, Ecclesiastes, Ecclesiasticus, Book of Wisdom) and four Asian (Upanishads, Yoga Sutras, Buddhist Sutras, Tao Te Ching) — represented by the counts of 8,266 words. The target is the book the passage comes from. The goal is to train a model that identifies the source book of an unseen passage from its vocabulary and to evaluate it honestly on data it never saw.

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

DATA_DIR = Path("data")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42

print("numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

## 2. Load and Inspect the Dataset

In [ ]:
raw = pd.read_csv(DATA_DIR / "AllBooks_baseline_DTM_Labelled.csv")
print("shape:", raw.shape)
raw.iloc[:5, :8]

In [ ]:
print("First column name:", repr(raw.columns[0]))
print("Example labels:", raw.iloc[:3, 0].tolist(), "...", raw.iloc[-2:, 0].tolist())
print()
print("Column data types:")
print(raw.dtypes.value_counts())
print()
print("Missing values in the whole table:", int(raw.isna().sum().sum()))

In [ ]:
counts = raw.iloc[:, 1:]
labels = raw.iloc[:, 0].str.split("_").str[0]
tokens = counts.sum(axis=1)

print("Passages per book (raw labels):")
print(labels.value_counts().to_string())
print()
print(f"sparsity (share of zero cells): {(counts == 0).to_numpy().mean():.4f}")
print(f"words that appear in only one passage: {int(((counts > 0).sum() == 1).sum()):,} of {counts.shape[1]:,}")
print(f"passages with zero words: {int((tokens == 0).sum())} -> {raw.loc[tokens == 0, raw.columns[0]].tolist()}")
print(f"words per passage: min {tokens.min()}, median {tokens.median():.0f}, max {tokens.max()}")
print()
print("median words per passage, by book:")
print(tokens.groupby(labels).median().sort_values().to_string())

**Data quality notes**

- The table has 590 rows and 8,266 columns: an unnamed first column with a label such as `Buddhism_Ch1`, and 8,266 integer word counts. There are no missing values and all counts are non-negative integers, so no imputation is needed.
- The target is not a column of its own; it has to be parsed from the label (the part before `_Ch`). One of the book names is misspelled in the source (`BookOfEccleasiasticus`).
- One passage (`Buddhism_Ch14`) contains no words at all: it carries no information and is removed.
- The classes are strongly imbalanced, from 189 passages of the Yoga Sutras down to 12 of Ecclesiastes. Accuracy alone would reward a model that ignores the small biblical books.
- The matrix is extremely sparse (99.1 % zeros) and 3,872 words occur in a single passage, which is a high-dimensional, small-sample setting (8,266 features for 589 usable passages) where regularization matters.